In [18]:
import os

# 创建 data 目录
os.makedirs(os.path.join('.', 'data'), exist_ok=True)
data_file = os.path.join('.', 'data', 'house_tiny.csv')

# 写入测试数据
with open(data_file, 'w', encoding='utf-8') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名：房间数、小巷类型、价格
    f.write('NA,Pave,127500\n')        # NA 表示缺失值
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [19]:
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


In [20]:
# 拆分输入（前两列）和输出（最后一列）
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]

# 1. 对数值型列（NumRooms）用均值填充
# 新版 Pandas 推荐先筛选出数值列，或者指定单列填充
numeric_cols = inputs.select_dtypes(include=['number']).columns
inputs[numeric_cols] = inputs[numeric_cols].fillna(inputs[numeric_cols].mean())

# 2. 对类别/字符串列（Alley）进行 One-Hot 编码（独热编码）
# dummy_na=True 会把 NaN 单独当作一个类别（如 Alley_nan）
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)

print("--- 处理后的 inputs ---")
print(inputs)

--- 处理后的 inputs ---
   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


In [21]:
import torch

# 明确指定数值类型为 float32，避免默认的 float64 导致 PyTorch 模型参数类型不匹配
X = torch.tensor(inputs.to_numpy(dtype=float), dtype=torch.float32)
y = torch.tensor(outputs.to_numpy(dtype=float), dtype=torch.float32)

print("--- PyTorch Tensors ---")
print("X:\n", X)
print("y:\n", y)

--- PyTorch Tensors ---
X:
 tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]])
y:
 tensor([127500., 106000., 178100., 140000.])
